<img src="https://raw.githubusercontent.com/harmonize-tools/socio4health/main/docs/source/_static/image.png" alt="socio4health logo" height="100" width="100"/>

# Hands-on with socio4health: preparing housing-deficit inputs from Colombia's Quality of Life Survey

**Run the tutorial via free cloud platforms:** [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/harmonize-tools/socio4health/HEAD?urlpath=%2Fdoc%2Ftree%2Fnotebooks%2Fexample_housing_deficit.ipynb) <a target="_blank" href="https://colab.research.google.com/github/harmonize-tools/socio4health/blob/main/notebooks/example_housing_deficit.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This notebook provides a real-world example of how to use **socio4health** to **retrieve** and **prepare** housing and household microdata from Colombia's Quality of Life Survey (**ECV**) for 2010–2017. The workflow creates annual input tables for the subsequent calculation of Colombia's housing deficit. It was developed as part of the data-preparation exercise associated with the unpublished manuscript *An Aedes-borne Diseases Vulnerability Indicator: Encompassing Viral Suitability, Socioeconomic Sensitivity, and Lack of Health System Resilience in the Latin America and the Caribbean Region and Colombia* by Umaña-Caro, J, et. al. Because the manuscript has not yet been published, no journal citation or DOI is provided. This tutorial assumes an **intermediate** or **advanced** understanding of **Python** and data manipulation.

## Setting up the environment

To run this notebook, you need to have the following prerequisite:

- **Python 3.10+**

Install `socio4health` with its scraping dependencies using ``pip``:

In [ ]:
!pip install socio4health -q
!pip show socio4health

## 1. Import libraries

The `Extractor` class from **socio4health** retrieves and reads the ECV files published by DANE. `pandas` is used to manipulate the resulting DataFrames, while `reduce` supports repeated merges by household directory identifier. `Path` and `uuid` are used to create a separate output directory for each run.

In [2]:
from socio4health import Extractor
from functools import reduce
import pandas as pd
from pathlib import Path
import uuid

## 2. Configure the ECV data

The `ecv_data` dictionary maps each year from 2010 to 2017 to its corresponding DANE microdata download page. The `vivienda_cols` and `hogar_cols` lists define the identifiers, geographic fields, and housing- and household-level variables retained by this workflow.

Two dictionaries, `dfs_vivienda_by_year` and `dfs_hogar_by_year`, are initialized to store the processed DataFrames separately for each year and unit of observation.

In [3]:
ecv_data = {
    2010: "https://microdatos.dane.gov.co/index.php/catalog/201/get-microdata",
    2011: "https://microdatos.dane.gov.co/index.php/catalog/196/get-microdata",
    2012: "https://microdatos.dane.gov.co/index.php/catalog/124/get-microdata",
    2013: "https://microdatos.dane.gov.co/index.php/catalog/213/get-microdata",
    2014: "https://microdatos.dane.gov.co/index.php/catalog/342/get-microdata",
    2015: "https://microdatos.dane.gov.co/index.php/catalog/419/get-microdata",
    2016: "https://microdatos.dane.gov.co/index.php/catalog/456/get-microdata",
    2017: "https://microdatos.dane.gov.co/index.php/catalog/544/get-microdata"
}

vivienda_cols = [
    "DIRECTORIO", "YEAR", "CLASE", "REGION", "P1_DEPARTAMENTO", "P1_MUNICIPIO",
    "P4000", "P4005", "P4015", "P8520S1", "P8520S3", "P8520S4", "P8520S5", "P70", "CANT_HOGARES_VIVIENDA"
]

hogar_cols = [
    "DIRECTORIO", "YEAR", "CLASE", "REGION", "P1_DEPARTAMENTO", "P1_MUNICIPIO",
    "P205", "CANT_PERSONAS_HOGAR", "P5010", "P8526", "P8530", "P8532", "P764"
]

dfs_vivienda_by_year = {}
dfs_hogar_by_year = {}

## 3. Extract housing and household data by year

Use the `Extractor` class to retrieve the `.sav` and `.zip` files required for every ECV edition. Because the archive names vary across years, year-specific regular expressions identify files related to housing identification, housing characteristics, household services, and household composition.

The `Extractor` is configured with the following parameters:

- `input_path`: the DANE microdata download page for the corresponding year.
- `down_ext`: the file extensions accepted by the extraction process.
- `output_path`: the local directory where the files for each year are downloaded and processed.
- `key_words`: regular expressions used to select the relevant files.
- `depth`: the number of link levels traversed from the supplied page.
- `delete_zip_after`: whether downloaded ZIP files are removed after extraction.

After extraction, column names are standardized to uppercase and known naming differences are reconciled. Each DataFrame is classified as housing- or household-level according to the variables it contains. Tables from the same year are then merged on `DIRECTORIO`, filtered to the configured columns, and stored for subsequent processing.

In [4]:
for year, url in ecv_data.items():
        print(f"\n{year}: {url}")
        
        if year == 2010:
            key_words = [
                r"(?i)datos[\s_]+(?:de[\s_]+)?identificaci[oó]n",
                r"(?i)servicios[\s_]+(?:del[\s_]+)?hogar",
                r"(?i)composici[oó]n[\s_]+(?:del[\s_]+)?hogar",
                r"(?i)datos[\s_]+(?:de[\s_]+)?(?:la[\s_]+)?vivienda",
            ]
        elif year == 2011:
            key_words = [
                r"(?i)identificaci[oó]n.*vivienda",
                r"(?i)vivienda.*identificaci[oó]n",
                r"(?i)datos[\s_]+de[\s_]+identificaci[oó]n",
                r"(?i)datos[\s_]+de[\s_]+la[\s_]+vivienda",
                r"(?i)servicios[\s_]+(?:del[\s_]+)?hogar",
            ]
        else:
            key_words = [
                r"(?i)datos[\s_]+(?:de[\s_]+)?identificaci[oó]n",
                r"(?i)servicios[\s_]+(?:del[\s_]+)?hogar",
                r"(?i)composici[oó]n[\s_]+(?:del[\s_]+)?hogar",
                r"(?i)datos[\s_]+(?:de[\s_]+)?(?:la[\s_]+)?vivienda",
            ]
        
        extractor = Extractor(
            input_path=url,
            down_ext=['.sav', '.zip'],
            sep=' ',
            output_path = f"data/example_housing_deficit/raw/ECV_{year}",
            depth=0,
            key_words=key_words,
            delete_zip_after=True
        )

        df_extracted = extractor.s4h_extract()
        
        dfs_vivienda_year = []
        dfs_hogar_year = []
        
        for df in df_extracted:
            df.columns = df.columns.str.strip()
            df.columns = [col.upper() for col in df.columns]
            df['YEAR'] = year
            
            # Normalizar nombres
            rename_map = {}
            if 'CANT_HOGARES_VIVIENDA' not in df.columns and 'CANT_HOG_COMPLETOS' in df.columns:
                rename_map['CANT_HOG_COMPLETOS'] = 'CANT_HOGARES_VIVIENDA'
            if rename_map:
                df = df.rename(columns=rename_map)
            
            # Renombrar columnas
            if 'DPTO' in df.columns and 'P1_DEPARTAMENTO' not in df.columns:
                df = df.rename(columns={'DPTO': 'P1_DEPARTAMENTO'})
            if 'P3' in df.columns and 'CLASE' not in df.columns:
                df = df.rename(columns={'P3': 'CLASE'})
            
            # Determinar si es tabla de vivienda o hogar
            has_vivienda_vars = any(col in df.columns for col in ['P4000', 'P4005', 'P4015', 'P8520'])
            has_hogar_vars = any(col in df.columns for col in ['P205', 'P5010', 'P8526', 'P8530', 'P8532', 'CANT_PERSONAS_HOGAR'])
            
            if 'DIRECTORIO' not in df.columns or df['DIRECTORIO'].isna().all():
                continue
            
            if has_vivienda_vars and not has_hogar_vars:
                dfs_vivienda_year.append(df)
            elif has_hogar_vars:
                dfs_hogar_year.append(df)
            elif has_vivienda_vars and has_hogar_vars:
                dfs_vivienda_year.append(df)
        
        # Merge de tablas de vivienda del mismo año
        if dfs_vivienda_year:
            if len(dfs_vivienda_year) == 1:
                merged_vivienda = dfs_vivienda_year[0]
            else:
                common_cols = set(dfs_vivienda_year[0].columns)
                for df in dfs_vivienda_year[1:]:
                    common_cols = common_cols.intersection(set(df.columns))
                common_cols.discard('DIRECTORIO')
                
                for i in range(1, len(dfs_vivienda_year)):
                    cols_to_drop = [col for col in common_cols if col in dfs_vivienda_year[i].columns]
                    if cols_to_drop:
                        dfs_vivienda_year[i] = dfs_vivienda_year[i].drop(columns=cols_to_drop)
                
                merged_vivienda = reduce(
                    lambda left, right: pd.merge(left, right, on='DIRECTORIO', how='outer'), 
                    dfs_vivienda_year
                )
            
            # Filtrar solo columnas de vivienda
            merged_vivienda = merged_vivienda[[col for col in vivienda_cols if col in merged_vivienda.columns]]
            dfs_vivienda_by_year[year] = merged_vivienda
            print(f"  -> Vivienda {year}: {len(merged_vivienda)} viviendas, columnas: {list(merged_vivienda.columns)}")
        
        # Merge de tablas de hogar del mismo año
        if dfs_hogar_year:
            if len(dfs_hogar_year) == 1:
                merged_hogar = dfs_hogar_year[0]
            else:
                common_cols = set(dfs_hogar_year[0].columns)
                for df in dfs_hogar_year[1:]:
                    common_cols = common_cols.intersection(set(df.columns))
                common_cols.discard('DIRECTORIO')
                
                for i in range(1, len(dfs_hogar_year)):
                    cols_to_drop = [col for col in common_cols if col in dfs_hogar_year[i].columns]
                    if cols_to_drop:
                        dfs_hogar_year[i] = dfs_hogar_year[i].drop(columns=cols_to_drop)
                
                merged_hogar = reduce(
                    lambda left, right: pd.merge(left, right, on='DIRECTORIO', how='outer'), 
                    dfs_hogar_year
                )
            
            # Filtrar solo columnas de hogar
            merged_hogar = merged_hogar[[col for col in hogar_cols if col in merged_hogar.columns]]
            dfs_hogar_by_year[year] = merged_hogar
            print(f"  -> Household {year}: {len(merged_hogar)} households, columns: {list(merged_hogar.columns)}")


2010: https://microdatos.dane.gov.co/index.php/catalog/201/get-microdata
15:38:22 INFO: Starting data extraction...
15:38:22 INFO: Extracting data in online mode...
15:38:22 INFO: Scraping URL: https://microdatos.dane.gov.co/index.php/catalog/201/get-microdata with depth 0
15:38:24 INFO: Found 4 files to download
15:38:24 INFO: Downloading 4 files to 'data/example_housing_deficit/raw/ECV_2010'
Downloading: 100%|██████████| 4/4 [00:01<00:00,  2.92file/s]                                                    
15:38:26 INFO: Downloaded 4/4 files
Extracting: 100%|██████████| 4/4 [00:00<00:00, 88.97archive/s]                                     
15:38:26 INFO: Extracted 4 files from 4 archives
Reading: 100%|██████████| 4/4 [00:00<00:00, 12.82file/s]                                                                      
15:38:26 INFO: Read 4/4 files
15:38:26 INFO: Deleted 4 archives after extraction
15:38:26 INFO: Extraction completed successfully.
  -> Vivienda 2010: 14268 viviendas, columnas:

## 4. Extract expansion factors

Survey expansion factors are needed to produce population-level estimates from the sampled households. For each available year, this step searches for the file whose name identifies expansion factors based on the 2018 National Population and Housing Census (**CNPV 2018**).

Only `DIRECTORIO` and `FEX_C_2018` are retained, with one expansion-factor record per directory identifier. If the expected file or columns are unavailable for a year, the exception is reported and the workflow continues with the remaining years.

In [5]:
dfs_fex_by_year = {}
for year, url in ecv_data.items():
    if 2010 <= year <= 2018:
        print(f"\n{year}: Extracting of Expansion Factors...")
        key_words_fex_2018 = [
            r"(?i)factores[\s_]+de[\s_]+expansi[oó]n[\s_]+(?:20)?[0-9]{2}[\s_]+basados[\s_]+en[\s_]+el[\s_]+CNPV[\s_]+2018",
        ]

        extractor_fex_2018 = Extractor(
            input_path=url,
            down_ext=['.sav', '.zip'],
            sep=' ',
            output_path = f"data/example_housing_deficit/raw/ECV_FEX_2018_{year}",
            depth=0,
            key_words=key_words_fex_2018,
            delete_zip_after=True
        )

        try:
            df_extracted_fex_2018 = extractor_fex_2018.s4h_extract()
            if df_extracted_fex_2018:
                df_fex = df_extracted_fex_2018[0]
                df_fex.columns = df_fex.columns.str.strip()
                df_fex.columns = [col.upper() for col in df_fex.columns]
                
                if 'DIRECTORIO' in df_fex.columns and 'FEX_C_2018' in df_fex.columns:
                    df_fex = df_fex[['DIRECTORIO', 'FEX_C_2018']].drop_duplicates(subset=['DIRECTORIO'])
                    dfs_fex_by_year[year] = df_fex
                    print(f"  -> FEX {year}: {len(df_fex)} registers")
        except Exception as e:
            print(f"Skipping year {year} for FEX_C_2018 extraction: {e}")


2010: Extracting of Expansion Factors...
15:38:52 INFO: Starting data extraction...
15:38:52 INFO: Extracting data in online mode...
15:38:52 INFO: Scraping URL: https://microdatos.dane.gov.co/index.php/catalog/201/get-microdata with depth 0
15:38:55 INFO: Found 1 files to download
15:38:55 INFO: Downloading 1 files to 'data/example_housing_deficit/raw/ECV_FEX_2018_2010'
Downloading: 100%|██████████| 1/1 [00:00<00:00,  3.39file/s]                                                               
15:38:55 INFO: Downloaded 1/1 files
Extracting: 100%|██████████| 1/1 [00:00<00:00, 90.86archive/s]                                                
15:38:55 INFO: Extracted 1 files from 1 archives
Reading: 100%|██████████| 1/1 [00:00<00:00, 26.64file/s]                                 
15:38:55 INFO: Read 1/1 files
15:38:55 INFO: Deleted 1 archives after extraction
15:38:55 INFO: Extraction completed successfully.
  -> FEX 2010: 14268 registers

2011: Extracting of Expansion Factors...
15:38:55 IN

## 5. Extract municipality identifiers

Some annual housing tables do not contain the municipality identifier directly. This step retrieves the ECV file describing the municipality where the survey was administered and retains the `DIRECTORIO`–`P1_MUNICIPIO` mapping. Duplicate directory identifiers are removed before the mapping is stored by year.

In [6]:
dfs_mpio_by_year = {}
for year, url in ecv_data.items():
    print(f"\n{year}: Extrancting Municipalities...")
    extractor_mpio = Extractor(
        input_path=url,
        down_ext=['.sav', '.zip'],
        sep=' ',
        output_path = f"data/example_housing_deficit/raw/ECV_MPIO_{year}",
        depth=0,
        key_words=[
            r"(?i)municipio[\s_]+de[\s_]+aplicaci[oó]n[\s_]+de[\s_]+la[\s_]+encuesta[\s_]+(?:20)?[0-9]{2}",
        ],
        delete_zip_after=True
    )

    try:
        df_extracted_mpio = extractor_mpio.s4h_extract()
        if df_extracted_mpio:
            df_mpio = df_extracted_mpio[0]
            df_mpio.columns = df_mpio.columns.str.strip()
            df_mpio.columns = [col.upper() for col in df_mpio.columns]
            
            if 'DIRECTORIO' in df_mpio.columns and 'P1_MUNICIPIO' in df_mpio.columns:
                df_mpio = df_mpio[['DIRECTORIO', 'P1_MUNICIPIO']].drop_duplicates(subset=['DIRECTORIO'])
                dfs_mpio_by_year[year] = df_mpio
                print(f"  -> Municipalities {year}: {len(df_mpio)} registers")
    except Exception as e:
        print(f"Skipping year {year} for municipio extraction: {e}")


2010: Extrancting Municipalities...
15:39:31 INFO: Starting data extraction...
15:39:31 INFO: Extracting data in online mode...
15:39:31 INFO: Scraping URL: https://microdatos.dane.gov.co/index.php/catalog/201/get-microdata with depth 0


15:39:36 WARNING: No downloadable files found matching criteria. Returning empty extraction.


15:39:36 INFO: Extraction completed successfully.


15:39:36 WARNING: No data was extracted. The extraction process returned an empty result.



2011: Extrancting Municipalities...
15:39:36 INFO: Starting data extraction...
15:39:36 INFO: Extracting data in online mode...
15:39:36 INFO: Scraping URL: https://microdatos.dane.gov.co/index.php/catalog/196/get-microdata with depth 0


15:39:38 WARNING: No downloadable files found matching criteria. Returning empty extraction.


... (22 outputs left out)


2017: Extrancting Municipalities...
15:39:48 INFO: Starting data extraction...
15:39:48 INFO: Extracting data in online mode...
15:39:48 INFO: Scraping URL: https://microdatos.dane.gov.co/index.php/catalog/544/get-microdata with depth 0


15:39:57 WARNING: No downloadable files found matching criteria. Returning empty extraction.


15:39:57 INFO: Extraction completed successfully.


15:39:57 WARNING: No data was extracted. The extraction process returned an empty result.


## 6. Extract additional 2010 identification fields

The 2010 ECV archive requires an additional extraction because its identification fields are distributed differently from those in later editions. The configured identification file is retrieved separately, and its column names are standardized so that department, area class, and region can be incorporated during consolidation.

In [7]:
dfs_2010_id = []
for year, url in ecv_data.items():
    if 2010 == year:
        print(f"\n{year}: Extracting <<identificacion-final>>...")
        key_words_2010_id = ["Datos de idenrificacion-final"]
        extractor_2010_id = Extractor(
            input_path=url,
            down_ext=['.sav', '.zip'],
            sep=' ',
            output_path = f"data/example_housing_deficit/raw/ECV_2010_ID_{year}",
            depth=0,
            key_words=key_words_2010_id,
            delete_zip_after=True
        )

        try:
            dfs_2010_id = extractor_2010_id.s4h_extract()
            for df in dfs_2010_id:
                df.columns = df.columns.str.strip()
                df.columns = [col.upper() for col in df.columns]
        except Exception as e:
            print(f"Skipping year {year} for 2010_ID extraction: {e}")


2010: Extracting <<identificacion-final>>...
15:39:57 INFO: Starting data extraction...
15:39:57 INFO: Extracting data in online mode...
15:39:57 INFO: Scraping URL: https://microdatos.dane.gov.co/index.php/catalog/201/get-microdata with depth 0
15:39:59 INFO: Found 1 files to download
15:39:59 INFO: Downloading 1 files to 'data/example_housing_deficit/raw/ECV_2010_ID_2010'
Downloading: 100%|██████████| 1/1 [00:00<00:00,  3.72file/s]                                          
15:39:59 INFO: Downloaded 1/1 files
Extracting: 100%|██████████| 1/1 [00:00<00:00, 83.33archive/s]                           
15:39:59 INFO: Extracted 1 files from 1 archives
Reading: 100%|██████████| 1/1 [00:00<00:00, 57.13file/s]                                                    
15:40:00 INFO: Read 1/1 files
15:40:00 INFO: Deleted 1 archives after extraction
15:40:00 INFO: Extraction completed successfully.


## 7. Consolidate the annual datasets

For each year, merge the expansion factor into the household table using `DIRECTORIO`. When the municipality is absent from the housing table, add it from the annual municipality mapping. For 2010, use the separately extracted identification table to complete `P1_DEPARTAMENTO`, `CLASE`, and `REGION` where those fields are available.

All merges in this step preserve the housing or household records already assembled for the year and add only the supporting fields needed for the final annual files.

In [8]:
for year in ecv_data.keys():
        print(f"\nProcessing {year}...")
        
        # Merge with expansion factor (only in df_hogar)
        if year in dfs_hogar_by_year and year in dfs_fex_by_year:
            df_hogar = dfs_hogar_by_year[year]
            df_fex = dfs_fex_by_year[year]
            
            df_hogar = df_hogar.merge(df_fex, on='DIRECTORIO', how='left')
            dfs_hogar_by_year[year] = df_hogar
            print(f"  -> Merge FEX_C_2018 finished in df_hogar")
        
        # Merge with municipality
        if year in dfs_vivienda_by_year and year in dfs_mpio_by_year:
            df_vivienda = dfs_vivienda_by_year[year]
            df_mpio = dfs_mpio_by_year[year]
            
            if 'P1_MUNICIPIO' not in df_vivienda.columns or df_vivienda['P1_MUNICIPIO'].isna().all():
                df_vivienda = df_vivienda.merge(df_mpio, on='DIRECTORIO', how='left')
                dfs_vivienda_by_year[year] = df_vivienda
                print(f"  -> Merge P1_MUNICIPIO finished")
        
        # Merge with identification 2010
        if year == 2010 and dfs_2010_id:
            df_2010_id = dfs_2010_id[0]
            
            if 'P3' in df_2010_id.columns and 'CLASE' not in df_2010_id.columns:
                df_2010_id = df_2010_id.rename(columns={'P3': 'CLASE'})
            
            rename_map = {}
            if 'P1_DEPARTAMENTO' not in df_2010_id.columns and 'DEPARTAMENTO' in df_2010_id.columns:
                rename_map['DEPARTAMENTO'] = 'P1_DEPARTAMENTO'
            if rename_map:
                df_2010_id = df_2010_id.rename(columns=rename_map)
            
            if 'DIRECTORIO' in df_2010_id.columns:
                df_2010_id = df_2010_id.drop_duplicates(subset=['DIRECTORIO'], keep='first')
                
                cols_to_merge = ['DIRECTORIO']
                for col in ['P1_DEPARTAMENTO', 'CLASE', 'REGION']:
                    if col in df_2010_id.columns:
                        cols_to_merge.append(col)
                
                if len(cols_to_merge) > 1 and year in dfs_vivienda_by_year:
                    df_vivienda = dfs_vivienda_by_year[year]
                    
                    df_vivienda = df_vivienda.merge(
                        df_2010_id[cols_to_merge], 
                        on='DIRECTORIO', 
                        how='left', 
                        suffixes=('', '_2010')
                    )
                    
                    for col in ['P1_DEPARTAMENTO', 'CLASE', 'REGION']:
                        col_2010 = f'{col}_2010'
                        if col_2010 in df_vivienda.columns:
                            if col in df_vivienda.columns:
                                df_vivienda[col] = df_vivienda[col].combine_first(df_vivienda[col_2010])
                            else:
                                df_vivienda[col] = df_vivienda[col_2010]
                            df_vivienda = df_vivienda.drop(columns=[col_2010])
                    
                    dfs_vivienda_by_year[year] = df_vivienda
                    print(f"  -> Merge identificacion 2010 finished")


Processing 2010...
  -> Merge FEX_C_2018 finished in df_hogar
  -> Merge identificacion 2010 finished

Processing 2011...
  -> Merge FEX_C_2018 finished in df_hogar

Processing 2012...
  -> Merge FEX_C_2018 finished in df_hogar

Processing 2013...
  -> Merge FEX_C_2018 finished in df_hogar

Processing 2014...
  -> Merge FEX_C_2018 finished in df_hogar

Processing 2015...
  -> Merge FEX_C_2018 finished in df_hogar

Processing 2016...
  -> Merge FEX_C_2018 finished in df_hogar

Processing 2017...
  -> Merge FEX_C_2018 finished in df_hogar


## 8. Save the annual CSV files

Create a unique run directory under `data/example_housing_deficit/output` so that a new execution does not overwrite previous results. Housing and household tables are combined with an outer merge on `DIRECTORIO`, the duplicated year fields are reconciled, and numeric columns are converted to appropriate pandas numeric types.

One file named `df_ecv_<year>.csv` is written for each successfully processed year. After the outputs are saved, the temporary ECV download directory is removed with `s4h_delete_download_folder`.

In [9]:
run_dir = Path("data/example_housing_deficit/output") / f"run_{uuid.uuid4().hex[:8]}"
run_dir.mkdir(parents=True, exist_ok=False)

def convert_dtypes(df):
    for col in df.columns:
        converted = pd.to_numeric(df[col], errors='coerce')
        non_null = converted.dropna()

        if len(non_null) > 0 and (non_null % 1 == 0).all():
            df[col] = converted.astype("Int64")
        else:
            df[col] = converted

    return df

years = sorted(set(dfs_vivienda_by_year.keys()) | set(dfs_hogar_by_year.keys()))
for year in years:
    df_vivienda = dfs_vivienda_by_year.get(year)
    df_hogar = dfs_hogar_by_year.get(year)

    if df_vivienda is not None and df_hogar is not None:
        df_joined = df_vivienda.merge(
            df_hogar,
            on='DIRECTORIO',
            how='outer',
            suffixes=('_vivienda', '_hogar')
        )

        if 'YEAR_vivienda' in df_joined.columns:
            df_joined = df_joined.rename(columns={'YEAR_vivienda': 'YEAR'})
        if 'YEAR_hogar' in df_joined.columns:
            if 'YEAR' in df_joined.columns:
                df_joined['YEAR'] = df_joined['YEAR'].combine_first(df_joined['YEAR_hogar'])
            else:
                df_joined = df_joined.rename(columns={'YEAR_hogar': 'YEAR'})
            df_joined = df_joined.drop(columns=['YEAR_hogar'])
    elif df_vivienda is not None:
        df_joined = df_vivienda.copy()
    elif df_hogar is not None:
        df_joined = df_hogar.copy()
    else:
        continue

    df_joined = convert_dtypes(df_joined)

    output_file = run_dir / f"df_ecv_{year}.csv"
    df_joined.to_csv(output_file, index=False)
    print(f"  -> {year}: {len(df_joined)} filas guardadas en {output_file.name}")

print(f"\nProcess completed. Files available at: {run_dir}")


if extractor is not None:
    extractor.s4h_delete_download_folder(folder_path="data/example_housing_deficit/raw")

  -> 2010: 14801 filas guardadas en df_ecv_2010.csv
  -> 2011: 25364 filas guardadas en df_ecv_2011.csv
  -> 2012: 21383 filas guardadas en df_ecv_2012.csv
  -> 2013: 21565 filas guardadas en df_ecv_2013.csv
  -> 2014: 20141 filas guardadas en df_ecv_2014.csv
  -> 2015: 23005 filas guardadas en df_ecv_2015.csv
  -> 2016: 22893 filas guardadas en df_ecv_2016.csv
  -> 2017: 8612 filas guardadas en df_ecv_2017.csv

Process completed. Files available at: data\example_housing_deficit\output\run_a974e0b7
15:40:00 INFO: Deleting folder: data\example_housing_deficit\raw
15:40:00 INFO: Folder deleted successfully


## Output and scope

The generated annual CSV files provide the prepared housing and household inputs for downstream housing-deficit calculations. This notebook does not calculate the housing deficit, construct the vulnerability indicator, or reproduce analytical results from the unpublished manuscript.